# 0. Environment Setup

In [1]:
# Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import sys
sys.path.insert(0, "..")

from src.config import (RAW_DATA_DIR, ROOT_DIR, PROCESSED_DATA_DIR, MODELS_DIR)

# Show all columns 
pd.set_option('display.max_columns', None)

# Bold Text Setup
b_start = '\033[1m'
b_end = '\033[0m'

# Graph visual style
sns.set_theme(style="whitegrid")

print('Environment Ready')

C:\Users\ASUS\.conda\envs\Personal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment Ready


-----
# 1. Complaint Narrative NLP

In [15]:
# Cleaned Dataset Load
print('Dataframe Loading...')

data = pd.read_csv(fr'{PROCESSED_DATA_DIR}\cleaned_complaints.csv', low_memory=False, delimiter=';')

print('Dataframe Correctly Loaded')
print()
display(data.sample(3))

Dataframe Loading...
Dataframe Correctly Loaded



,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id,month_received
106739,2021-06-12,Checking or savings account,Checking account,Managing an account,Fee problem,i contacted navy federal on numerous occasions...,NAVY FEDERAL CREDIT UNION,MS,39212,Servicemember,Web,2021-06-12,Closed with monetary relief,Yes,4455537,2021-06-01
9661,2021-04-04,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,i rented a car today on and three weeks later ...,CHIME FINANCIAL INC,MI,48219,NaN,Web,2021-04-04,Closed with explanation,Yes,4270289,2021-04-01
34975,2020-01-18,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,on the my car got broken into whiles i was wor...,BANK OF AMERICA NATIONAL ASSOCIATION,WA,98203,NaN,Web,2020-01-19,Closed with monetary relief,Yes,3501579,2020-01-01


In [17]:
# Embedding model load/training
MODEL_PATH = fr'{MODELS_DIR}\bertopic_complaints_model'

if os.path.exists(MODEL_PATH):
    print(f"Model found in {MODELS_DIR}. Loading...")
    topic_model = BERTopic.load(MODEL_PATH)
    print("Model succesfully loaded")

    topics, probs = topic_model.transform(data['consumer_complaint_narrative'].to_list())

    data['topic'] = topics
    data['topic_prob'] = probs
    
else:
    print("Model not found. Training new model...")
    # Embedding model
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    
    # Stop words clean and tokenization 
    vectorizer_model = CountVectorizer(stop_words="english")
    analyzer = vectorizer_model.build_analyzer()
    
    data['tokenized_text'] = data['consumer_complaint_narrative'].apply(analyzer)
    
    # BERTopic Initialization and Training
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        nr_topics="auto", # Automatically reduces similar topics
        verbose=True
    )
    
    # Model adjustment and topic prediction
    topics, probs = topic_model.fit_transform(data['consumer_complaint_narrative'].to_list())
    
    data['topic'] = topics
    data['topic_prob'] = probs

    # Model save
    topic_model.save(MODEL_PATH)
    print(f'Model saved to folder {MODELS_DIR}')

display(data.sample(3))

Model found in C:\Users\ASUS\OneDrive\Desktop\Proyectos\Consumer-Complaints-Fraud-Correlation-Analysis\notebooks\..\models. Loading...
Model succesfully loaded


Batches: 100%|██████████| 3893/3893 [38:12<00:00,  1.70it/s]   


,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id,month_received,topic,topic_prob
37200,2020-03-17,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,Fraud or scam,NaN,i order a moving service to move me for per ho...,SQUARE INC,GA,30058,NaN,Web,2020-03-17,Closed with explanation,Yes,3569987,2020-03-01,8,0.943798
35809,2019-10-19,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Their investigation did not fix an error on yo...,this particular account situation that is late...,CAPITAL ONE FINANCIAL CORPORATION,MN,55117,NaN,Web,2019-10-21,Closed with explanation,Yes,3411123,2019-10-01,80,0.740734
5186,2020-03-27,Credit card or prepaid card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Confusing or misleading advertising about the ...,i received a promotion by mail for an american...,AMERICAN EXPRESS COMPANY,NY,13089,NaN,Web,2020-03-27,Closed with explanation,Yes,3582366,2020-03-01,-1,0.000000


In [18]:
# Suspicious terms exploration
suspicious_terms = [
    'unauthorized', 'stolen', 'card', 'identity', 'fraud', 
    'atm', 'phishing', 'unsolicited', 'deceptive', 'hack',
    'cash advance', 'scam', 'fake', 'without authorization',
    'unrecognized', 'never', 'error', 'abnormal', 'counterfeit',
    'forged', 'misrepresentation', 'bribe', 'corrupt', 'launder',
    'impersonate', 'impersonating', 'theft', 'hacked', 'breach',
    'unauthorised', 'fraudulent', 'cloned', 'clone', 'skimming'
    'suspicious', 'data leak', 'cyberattack', 'crime', 'criminal',
    'unknown', 'unexpected'
]

# Frequent topics
topics_df = pd.DataFrame(topic_model.get_topic_info())
topics_df.columns = topics_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

display(topics_df[topics_df['representation'].apply(lambda x: any(term in x for term in suspicious_terms))].sample(3))

# # Inspect words related to each topic
# print(topic_model.get_topic(3))

,topic,count,name,representation,representative_docs
104,103,131,103_citi_citibank_verify_number,"[citi, citibank, verify, number, verification,...",[summary citi has account verification setup p...
358,357,14,357_atm_withdrawal_bow_withdraw,"[atm, withdrawal, bow, withdraw, non, fee, inq...",[bank of the west has been double charging for...
333,332,17,332_hacked_computer_amounting_someones,"[hacked, computer, amounting, someones, hacked...",[someone i do not know hacked my computer and ...


In [20]:
# Enriched Dataframe Export
file_name = 'topic_enriched_complaints.csv'
data.to_csv(fr'{PROCESSED_DATA_DIR}\{file_name}', index=False, sep=';')
print(f'Dataframe {file_name} exported to folder {PROCESSED_DATA_DIR}')

# Topics Dataframe Export
file_name = 'topic_info.csv'
topics_df.to_csv(fr'{PROCESSED_DATA_DIR}\{file_name}', index=False, sep=';')
print(f'Dataframe {file_name} exported to folder {PROCESSED_DATA_DIR}')

Dataframe topic_enriched_complaints.csv exported to folder C:\Users\ASUS\OneDrive\Desktop\Proyectos\Consumer-Complaints-Fraud-Correlation-Analysis\notebooks\..\data\processed_data
Dataframe topic_info.csv exported to folder C:\Users\ASUS\OneDrive\Desktop\Proyectos\Consumer-Complaints-Fraud-Correlation-Analysis\notebooks\..\data\processed_data
